# 04 — Modelos Avanzados, Ensemble y Evaluación Ética
**CRISP-DM: Modelado + Evaluación** · Etapa 2: *hiperparámetros, modelos avanzados, CV rigurosa + prueba final, equidad/ética*

Sobre los baselines del notebook 03 (Regresión Lineal, Random Forest, XGBoost), esta etapa agrega:
1. Optimización de hiperparámetros de XGBoost (Optuna).
2. Modelos avanzados: Red Neuronal (MLP) y LSTM (Keras).
3. Ensemble (stacking) de los modelos tabulares.
4. Validación cruzada temporal rigurosa + prueba final única en un tramo de test que ningún modelo/ajuste vio antes.
5. Evaluación ética y de sesgos (equidad de desempeño entre productos/temporadas).

Requiere `pip install -r requirements.txt` + `tensorflow-cpu` + `optuna`.

In [ ]:
import sys, time
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
from prediccion_precios import config, features as ft, evaluation as ev
from prediccion_precios import models_baseline as mb, models_advanced as ma, fairness as fa
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)

## 1. Datos + split riguroso train / val / test

**Por qué 3 tramos y no 2:** `val` se usa para ajustar el ensemble (pesos/meta-modelo) sin tocar el test; `test` se evalúa **una sola vez**, al final, para que la tabla comparativa no esté inflada por haberse usado —ni directa ni indirectamente— durante el ajuste de ningún modelo. Es la misma idea del notebook 02 (nada de fuga de información), aplicada ahora a la selección de modelo.

In [ ]:
data = pd.read_csv(config.DATASET_MODELADO, parse_dates=[config.COL_FECHA])
X, y = ft.construir_matriz_modelado(data)
X = X.astype(float)
X_train, X_val, X_test, y_train, y_val, y_test = ev.split_train_val_test(X, y)
print("train", len(X_train), "| val", len(X_val), "| test", len(X_test))

train 510 | val 118 | test 157


## 2. Optimización de hiperparámetros — XGBoost (Optuna)

Búsqueda bayesiana (TPE) de 40 combinaciones de hiperparámetros, cada una evaluada con `TimeSeriesSplit` (ventana expansiva) **solo sobre `X_train`** — el test no participa en absoluto de esta búsqueda.

In [ ]:
t0 = time.time()
xgb_optimizado, info_hpo = ma.optimizar_xgboost(X_train, y_train, metodo="optuna", n_trials=40)
print(f"HPO en {time.time()-t0:.1f}s")
info_hpo

HPO en 259.9s


{'metodo': 'optuna',
 'n_trials': 40,
 'mejores_parametros': {'n_estimators': 200,
  'max_depth': 7,
  'learning_rate': 0.04641127794035395,
  'subsample': 0.9617511176758824,
  'colsample_bytree': 0.810960200314673,
  'min_child_weight': 7,
  'reg_lambda': 5.418002452500865},
 'mejor_mape_cv': 9.3633}

## 3. Modelos avanzados

**Red Neuronal (MLP):** mismas features tabulares que los baselines, con escalado estándar (necesario porque las variables mezclan escalas muy distintas) y early stopping monitoreando `val`.

**LSTM:** en vez de reusar las features de lags/medias móviles, toma directamente la *secuencia* de las 8 semanas previas de precio para predecir la semana siguiente — un enfoque distinto y más propio de una red recurrente. Se arma por producto (sin mezclar series) y se ordena por fecha para respetar el tiempo.

In [ ]:
t0 = time.time()
red_neuronal = ma.RedNeuronalRegresora(epochs=300, batch_size=16, paciencia=30, verbose=0)
red_neuronal.fit(X_train, y_train, X_val, y_val)
pred_nn_test = red_neuronal.predict(X_test)
print(f"MLP entrenado en {time.time()-t0:.1f}s | métricas test:", ev.calcular_metricas(y_test, pred_nn_test))

MLP entrenado en 24.3s | métricas test: {'MAE': np.float64(3.2223), 'RMSE': np.float64(4.0702), 'MAPE_%': np.float64(12.395), 'R2': np.float64(0.9815)}


In [ ]:
Xs, ys, fechas_s, productos_s = ma.preparar_secuencias(data, ventana=8, columnas=[config.COL_PRECIO])
print("secuencias:", Xs.shape)

n = len(Xs)
n_test_s = int(np.ceil(n * config.TEST_SIZE)); n_val_s = int(np.ceil(n * 0.15))
n_train_s = n - n_test_s - n_val_s
Xs_train, Xs_val, Xs_test = Xs[:n_train_s], Xs[n_train_s:n_train_s+n_val_s], Xs[n_train_s+n_val_s:]
ys_train, ys_val, ys_test = ys[:n_train_s], ys[n_train_s:n_train_s+n_val_s], ys[n_train_s+n_val_s:]
print("train/val/test secuencias:", len(Xs_train), len(Xs_val), len(Xs_test))

secuencias: (785, 8, 1)
train/val/test secuencias: 510 118 157


In [ ]:
t0 = time.time()
lstm = ma.LSTMRegresor(unidades=64, epochs=300, batch_size=16, paciencia=30, verbose=0)
lstm.fit(Xs_train, ys_train, Xs_val, ys_val)
pred_lstm_test = lstm.predict(Xs_test)
print(f"LSTM entrenado en {time.time()-t0:.1f}s | métricas test:", ev.calcular_metricas(ys_test, pred_lstm_test))

LSTM entrenado en 79.4s | métricas test: {'MAE': np.float64(1.4369), 'RMSE': np.float64(2.322), 'MAPE_%': np.float64(5.1212), 'R2': np.float64(0.994)}


> **Nota metodológica:** la LSTM usa una ventana de 8 semanas por producto, así que su tramo de test tiene una cantidad de filas ligeramente distinta al de los modelos tabulares (ambos parten de la misma rejilla, pero el reordenamiento por fecha entre productos con la misma semana no es idéntico al de `construir_matriz_modelado`). Por eso se reporta por separado y no se mezcla en la tabla comparativa de la sección 6, ni entra al ensemble tabular de la sección 4.

## 4. Ensemble (stacking)

Combina Regresión Lineal + Random Forest + XGBoost optimizado (los 3 modelos con interfaz tabular estándar) mediante un meta-modelo Ridge, ajustado sobre las predicciones en `X_val` — nunca sobre `X_train` (para que el meta-modelo no aprenda sobre datos que los modelos base ya memorizaron) ni sobre `X_test`.

In [ ]:
lin = mb.entrenar_regresion_lineal(X_train, y_train)
rf = mb.entrenar_random_forest(X_train, y_train)
modelos_tabulares = {"regresion_lineal": lin, "random_forest": rf, "xgboost_optimizado": xgb_optimizado}
ensemble = ma.construir_ensemble(modelos_tabulares, X_val, y_val, metodo="stacking")
pred_ensemble_test = ensemble.predict(X_test)
ev.calcular_metricas(y_test, pred_ensemble_test)

{'MAE': np.float64(1.5169),
 'RMSE': np.float64(2.3709),
 'MAPE_%': np.float64(6.4885),
 'R2': np.float64(0.9937)}

## 5. Validación cruzada temporal rigurosa (sobre train, ventana expansiva)

Chequeo de estabilidad de cada modelo entre folds (más allá del único número de HPO): ¿el error varía mucho entre ventanas de tiempo? Se usa solo `X_train`/`y_train` — el test sigue reservado.

In [ ]:
fabricas_cv = {
    "regresion_lineal": mb.nuevo_regresion_lineal,
    "random_forest": mb.nuevo_random_forest,
    "xgboost_optimizado": lambda: mb.nuevo_xgboost(**info_hpo["mejores_parametros"]),
}
for nombre, fabrica in fabricas_cv.items():
    cv = ev.validacion_cruzada_temporal(fabrica, X_train, y_train, n_splits=config.CV_SPLITS)
    print(nombre, "-> MAPE % (media ± std):", cv["MAPE_%"]["media"], "±", cv["MAPE_%"]["std"])

regresion_lineal -> MAPE % (media ± std): 97.873 ± 163.22


random_forest -> MAPE % (media ± std): 9.6741 ± 4.0985


xgboost_optimizado -> MAPE % (media ± std): 9.3633 ± 4.6088


## 6. Prueba final única — tabla comparativa completa

Cada modelo se evalúa **una sola vez** sobre `X_test`/`y_test`, el tramo que ningún ajuste (ni de hiperparámetros, ni del ensemble) llegó a ver.

In [ ]:
resultados = {
    "regresion_lineal": ev.calcular_metricas(y_test, lin.predict(X_test)),
    "random_forest": ev.calcular_metricas(y_test, rf.predict(X_test)),
    "xgboost_optimizado": ev.calcular_metricas(y_test, xgb_optimizado.predict(X_test)),
    "red_neuronal": ev.calcular_metricas(y_test, pred_nn_test),
    "ensemble_stacking": ev.calcular_metricas(y_test, pred_ensemble_test),
}
tabla_final = ev.tabla_comparativa(resultados)
print("Meta objetivo: MAPE <", config.META_MAPE_OBJETIVO * 100, "%")
print("(referencia, tramo distinto) LSTM MAPE %:", ev.calcular_metricas(ys_test, pred_lstm_test)["MAPE_%"])
tabla_final

Meta objetivo: MAPE <

 15.0 %
(referencia, tramo distinto) LSTM MAPE %: 5.1212


,MAE,RMSE,MAPE_%,R2
Modelo,,,,
regresion_lineal,1.4676,2.3513,5.9216,0.9938
ensemble_stacking,1.5169,2.3709,6.4885,0.9937
xgboost_optimizado,1.9519,3.1321,7.4932,0.9890
random_forest,1.8836,3.1315,7.6657,0.9890
red_neuronal,3.2223,4.0702,12.3950,0.9815


In [ ]:
modelo_final_nombre = tabla_final.index[0]
print("Modelo elegido para despliegue (mejor MAPE en la prueba final):", modelo_final_nombre)

Modelo elegido para despliegue (mejor MAPE en la prueba final): regresion_lineal


## 7. Evaluación ética y de sesgos

Un MAPE global bajo puede esconder que el modelo es mucho peor para ciertos productos o temporadas. Se usa el modelo elegido en la sección anterior.

In [ ]:
predicciones_finales = {
    "regresion_lineal": lin.predict(X_test), "random_forest": rf.predict(X_test),
    "xgboost_optimizado": xgb_optimizado.predict(X_test), "red_neuronal": pred_nn_test,
    "ensemble_stacking": pred_ensemble_test,
}[modelo_final_nombre]

grupo_cols = [c for c in X_test.columns if c.startswith("grupo_")]
grupo_test = X_test[grupo_cols].idxmax(axis=1).str.replace("grupo_", "", regex=False)

tabla_grupo = fa.metricas_por_grupo(y_test, predicciones_finales, grupo_test)
tabla_temporada = fa.metricas_por_temporada(y_test, predicciones_finales, X_test["es_cosecha"])
display(tabla_grupo)
display(tabla_temporada)

,n,MAE,RMSE,MAPE_%,R2
grupo,,,,,
ARROZ,31,0.4358,0.6753,0.9537,-13.2306
FRIJOL,32,2.1072,2.8470,2.2310,0.2095
MAÍZ,32,0.7346,0.8777,3.3745,0.7430
NARANJA,31,0.7145,0.9162,5.2779,0.6036
TOMATE,31,3.3488,4.1884,17.9720,0.6763


,n,MAE,RMSE,MAPE_%,R2
grupo,,,,,
no_cosecha,96,1.0443,1.5108,4.3912,0.9973
cosecha,61,2.1339,3.2615,8.3299,0.9893


In [ ]:
print(fa.reporte_texto(tabla_grupo, tabla_temporada))

Disparidad de error entre productos (peor/mejor MAPE): 18.84x
Disparidad de error cosecha vs. no cosecha (peor/mejor MAPE): 1.90x

Limitaciones y consideraciones éticas:
- Los precios son de mercados MAYORISTAS del MAG: no reflejan directamente lo
  que paga el consumidor final ni lo que recibe el pequeño productor (los
  márgenes de intermediación no están capturados). No usar la predicción como
  proxy del ingreso del productor ni del precio al consumidor sin ajuste.
- Cobertura acotada (5 productos, 2021-2024, mercados formales): el modelo no
  generaliza a otros cultivos, regiones o choques atípicos fuera del
  histórico (sequías extremas, crisis de combustible, cierres de frontera).
- Uso previsto: apoyo informativo (planificación de compras institucionales,
  alerta temprana de variabilidad de precios). No debe usarse para
  especulación, acaparamiento o fijación de precios que perjudique a
  productores o consumidores.
- Desempeño desigual entre productos y temporadas (ver `metr

## 8. Guardar modelos entrenados (para la API de despliegue)

Se guardan TODOS los modelos (no solo el elegido) para que la app pueda ofrecer comparación entre ellos. La red neuronal y la LSTM se guardan en formato Keras + escaladores por separado (no son serializables con `joblib` de forma confiable); el ensemble sí, porque sus 3 componentes son modelos scikit-learn/XGBoost estándar.

In [ ]:
import joblib

mb.guardar_modelo(xgb_optimizado, "xgboost_optimizado")
mb.guardar_modelo(ensemble, "ensemble_stacking")
red_neuronal.guardar(str(config.MODELS_DIR / "red_neuronal"))
lstm.guardar(str(config.MODELS_DIR / "lstm"))

config.ETAPA2_DIR.mkdir(parents=True, exist_ok=True)
tabla_final.to_csv(config.ETAPA2_DIR / "tabla_comparativa_etapa2.csv")
with open(config.ETAPA2_DIR / "hiperparametros_xgboost.json", "w", encoding="utf-8") as f:
    import json
    json.dump(info_hpo, f, indent=2, ensure_ascii=False)
with open(config.ETAPA2_DIR / "reporte_equidad.txt", "w", encoding="utf-8") as f:
    f.write(fa.reporte_texto(tabla_grupo, tabla_temporada))

print("Modelos guardados en", config.MODELS_DIR)
print("Modelo recomendado para la API:", modelo_final_nombre)

Modelos guardados en C:\Users\Ponce\Documents\Maching_Learning_UES\prediccion_precios_agricolas\models
Modelo recomendado para la API: regresion_lineal


### Conclusiones de Etapa 2
> Comparar el mejor modelo avanzado/ensemble contra los baselines de Etapa 1 (¿mejoró el MAPE?), señalar qué producto/temporada concentra el mayor error (sección 7) y dejar explícitas las limitaciones éticas antes de usar el modelo en la app (Streamlit + Flask, siguiente paso).